# 01 · ArUco 마커 13개(id 0~12) 생성

목표: **id0을 기준점(원점)**으로 하고 id1~id12를 작업공간에 무작위 배치할 마커 13개를 인쇄용으로 생성한다.
다음 노트북(`02_markers_to_3d.ipynb`)에서 사진 한 장을 찍어 이 마커들을 검출하고 id0 기준 3D 좌표로 가상공간에 표시한다.

> **인쇄 규칙:** 13개를 **모두 같은 물리 크기**로 인쇄하고, 인쇄 후 마커 한 변(검은 테두리 바깥 정사각형)을 자로 재서 `02` 노트북의 `MARKER_LENGTH_M`에 넣는다. 크기가 서로 다르면 스케일이 틀어진다.

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

print('OpenCV', cv2.__version__)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
MARKERS_DIR = os.path.join(ROOT, 'output', 'markers')
os.makedirs(MARKERS_DIR, exist_ok=True)

# 마커 설정
ARUCO_DICT = cv2.aruco.DICT_4X4_50   # 4x4, 50개 id (0~12 사용). 검출 쉬움
N_MARKERS = 13                        # id 0 ~ 12
SIDE_PX = 600                         # 개별 마커 이미지 해상도
BORDER_BITS = 1
dictionary = cv2.aruco.getPredefinedDictionary(ARUCO_DICT)
print(f'dictionary=DICT_4X4_50, 마커 {N_MARKERS}개 (id 0~{N_MARKERS-1})')

In [ ]:
# 개별 마커 PNG 저장 (흰 여백 포함해 인쇄/검출 안정화)
quiet = SIDE_PX // 8   # 마커 주위 흰 여백(quiet zone)
for mid in range(N_MARKERS):
    marker = cv2.aruco.generateImageMarker(dictionary, mid, SIDE_PX, borderBits=BORDER_BITS)
    canvas = np.full((SIDE_PX + 2 * quiet, SIDE_PX + 2 * quiet), 255, np.uint8)
    canvas[quiet:quiet + SIDE_PX, quiet:quiet + SIDE_PX] = marker
    # id 라벨(인쇄물에서 어느 게 어느 id인지 구분용) — 검출 영역 밖 여백에 표기
    cv2.putText(canvas, f'id{mid}', (10, quiet - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, 0, 2, cv2.LINE_AA)
    path = os.path.join(MARKERS_DIR, f'marker_{mid:02d}.png')
    cv2.imwrite(path, canvas)
print(f'개별 마커 {N_MARKERS}개 저장 →', MARKERS_DIR)

In [ ]:
# 한 장으로 보는 컨택트 시트 (미리보기 + 참고 인쇄용)
cols = 4
rows = int(np.ceil(N_MARKERS / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.6))
for ax in axes.flat:
    ax.axis('off')
for mid in range(N_MARKERS):
    m = cv2.aruco.generateImageMarker(dictionary, mid, 300, borderBits=BORDER_BITS)
    ax = axes.flat[mid]
    ax.imshow(m, cmap='gray')
    title = 'id0  (기준/원점)' if mid == 0 else f'id{mid}'
    ax.set_title(title, fontsize=11, color=('crimson' if mid == 0 else 'black'))
fig.suptitle('ArUco markers id0~id12 (DICT_4X4_50)', fontsize=13)
fig.tight_layout()
sheet_path = os.path.join(ROOT, 'output', 'markers_contact_sheet.png')
fig.savefig(sheet_path, dpi=150, bbox_inches='tight')
print('컨택트 시트 저장 →', sheet_path)
plt.show()

## 다음 단계

1. `output/markers/marker_00.png` ~ `marker_12.png`를 **모두 같은 크기로 인쇄**(또는 태블릿 화면에 표시). 두꺼운 종이에 붙이면 평평해서 검출이 안정적.
2. 인쇄한 마커 한 변을 자로 측정 → 미터로 기록 (예: 5cm → 0.05).
3. **id0을 기준점**에 두고 id1~id12를 작업공간에 배치.
4. 카메라로 **한 장** 찍어 `data/scene_images/`에 저장.
5. `02_markers_to_3d.ipynb` 실행 → id0 기준 3D 좌표로 가상공간에 표시.